In [ ]:
# Notebook 04: Performance Evaluation & Figures (All Methods - Silhouette)

import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"   # ground truth file

# folders for each method
imputed_dirs = {
    "SoftImpute": "imputed_softimpute",
    "MAGIC": "imputed_h5ad",
    "KNN": "imputed_knn",
    "Mean": "imputed_mean",
    "GAN": "imputed_gan",
    "Iterative": "imputed_iterative"
}

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    adata = adata.copy()
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)
    return adata

# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")

# -----------------------------
# Helper: Silhouette
# -----------------------------
def compute_silhouette(adata):
    try:
        score = silhouette_score(adata.obsm["X_pca"], adata.obs["leiden"])
    except Exception:
        score = np.nan
    return score

# -----------------------------
# Evaluate all methods
# -----------------------------
results = []

for method, imputed_dir in imputed_dirs.items():
    if not os.path.exists(imputed_dir):
        print(f"⚠️ Skipping {method}, folder not found: {imputed_dir}")
        continue

    for fname in os.listdir(imputed_dir):
        if not fname.endswith(".h5ad"):
            continue

        fpath = os.path.join(imputed_dir, fname)
        try:
            adata_imp = sc.read_h5ad(fpath)
            adata_imp = preprocess_adata(adata_imp)

            # Align cells
            common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
            adata_imp_sub = adata_imp[common_cells]

            # Compute Silhouette
            sil = compute_silhouette(adata_imp_sub)

            # Parse metadata (mfXX, runX)
            mf, run = None, None
            for part in fname.split("_"):
                if part.startswith("mf"):
                    mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
                if part.startswith("run"):
                    run = int(part.replace("run", "").replace(".h5ad", ""))

            results.append({
                "file": fname,
                "method": method,
                "Silhouette": sil,
                "missing_fraction": mf,
                "run": run
            })

            print(f"{method} - {fname}: Silhouette={sil:.3f}")

        except Exception as e:
            print(f"Error with {method} - {fname}: {e}")

# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["Silhouette", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Plot all methods with palette
# -----------------------------
palette = {
    "SoftImpute": "#1f77b4",   # blue
    "MAGIC": "#ff7f0e",        # orange
    "KNN": "#2ca02c",          # green
    "Mean": "#d62728",         # red
    "GAN": "#9467bd",          # purple
    "Iterative": "#8c564b"     # brown
}

plt.figure(figsize=(10, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="Silhouette",
    hue="method",
    errorbar="sd",
    capsize=0.1,
    palette=palette
)
plt.title("Silhouette Score across Missing Fractions (All Methods)")
plt.ylabel("Silhouette Score")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()
